# Lab 1 — Frame, Split, Baseline
**SDA-AIE-111 · Day 1 · Hour 3 · 50 minutes · pairs**

**Objective.** Produce the framing canvas, a verified stratified split, and committed dummy + heuristic baselines for both Manafeth tasks (churn classification, basket-value regression).

**The rules this lab installs for the whole week:**
1. **Split first** — the test set is quarantined before any modelling decision.
2. **Baseline always** — every later model must beat the dummy *and* the domain heuristic.
3. **The timestamp test** — a feature is legal only if its value exists at prediction time.

> ⏱ Task timings are in each section header. If you fall behind, the checkpoint solution notebook lets you fast-forward.


In [ ]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

## Task 1 — Data audit *(10 min)*

Load `manafeth_customers` and answer the three audit questions **in the markdown cell below the code**:
1. What is one row? (unit of analysis)
2. Is the target computable, and how balanced is it?
3. What can hurt us later? (missingness, weird distributions, suspicious columns)


In [ ]:
customers = pd.read_parquet(DATA / "manafeth_customers.parquet")
print(customers.shape)
customers.head(3)

In [ ]:
# TODO Audit question 1: assert that customer_id is unique (unit of analysis = ?)
# TODO Audit question 2: print the target balance (value_counts, normalize=True)
# TODO Audit question 3: print the top-5 missingness rates (isna().mean())


**Your audit answers (fill in):**
- One row = …
- Churn rate = … → remember this number; it explains Module 4
- Missingness: `avg_rating` … (who is missing it, and why is that *informative*?)


## Task 2 — Framing canvas + leak hunt *(10 min)*

Complete `project/FRAMING.md` in pairs (template in the repo). Then use the **data dictionary** (`data/data_dictionary.md`) and the timestamp test — *would this value exist at prediction time?* — to flag the leaky columns. There are **three planted leaks**; find at least two.


In [ ]:
# TODO: list the leaky columns you found and show evidence for the sneakiest one
# Hint: group next_month_orders by churned_30d and look at the means.
LEAKY = [...]  # fill in


## Task 3 — The split *(10 min)*

Stratified, seeded, quarantined. The split happens **now**, in cell ~6 — not cell 30.


In [ ]:
from sklearn.model_selection import train_test_split

TARGET = "churned_30d"
# TODO: build X (drop target, customer_id and the leaky columns) and y
# TODO: stratified 80/20 split with random_state=RANDOM_STATE
# TODO: verify the churn rate matches in both halves (print both means)
# TODO: save the test partition to DATA / "splits" / "test_quarantine.parquet"


### 🔒 DO NOT OPEN
The file `data/splits/test_quarantine.parquet` is now **quarantined**. It is read exactly once, in Lab 7, for the final evaluation. Every decision between now and then uses the training data only. A test set consulted ten times is a validation set with a misleading name.


## Task 4 — Baselines *(10 min)*

Two baselines per task. The dummy defines the floor; the heuristic is what the business could deploy this afternoon for free.


In [ ]:
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import average_precision_score, mean_absolute_error

def recall_at_budget(y_true, scores, budget=0.20):
    """Recall among the top `budget` share ranked by score — the framing-canvas metric."""
    n = int(budget * len(y_true))
    top = np.argsort(-np.asarray(scores))[:n]
    return np.asarray(y_true)[top].sum() / np.asarray(y_true).sum()

# TODO: DummyClassifier(strategy="stratified") -> PR-AUC on train
# TODO: the 21-day-silence heuristic = rank customers by days_since_last_order,
#       take the top 20%, report recall (use recall_at_budget)
# TODO: DummyRegressor(strategy="mean") on avg_basket_sar -> MAE in SAR


## Task 5 — Scoreboard *(5 min)*

Post your numbers to the class scoreboard. **They must match your neighbours' to two decimals** — same seed, same data, same workflow. Divergent numbers always mean a workflow bug, and finding it publicly is the lesson.

| Model | PR-AUC | recall@20% | MAE (SAR) |
|---|---|---|---|
| dummy (stratified) | | | — |
| heuristic (21-day silence) | — | | — |
| dummy regressor (mean) | — | — | |

## Task 6 — Commit *(5 min)*

```
git add -A && git commit -m "feat(lab1): framing canvas, quarantined split, baselines"
```
